# Marketplace Analytics Platform
## Notebook 2 — Data Quality Investigations & Validation Decisions

This notebook validates the trustworthiness of the Olist dataset through systematic checks on identifier consistency, referential integrity, temporal logic, and completeness. Every anomaly surfaced during assessment is investigated here, and each investigation closes with a documented decision on how the issue will be handled in the next phase of ETL pipeline.

In [1]:
import sys
sys.path.append("../python")

from utils.imports import *
from utils.load_data import load_datasets
from utils.date_conversion import convert_dates

dfs = load_datasets()
dfs = convert_dates(dfs)

---
### Investigations Triggered by Data Assessment

The following investigations were triggered by anomalies observed during structural profiling and date range assessment in `01_data_assessment.ipynb`.

- Investigating `shipping_limit_date` anomaly

#### **Investigating shipping_limit_date anomaly**

The overall date range check above showed `shipping_limit_date` extending to 2020 — about two years past the last actual purchase in the dataset (Oct 2018). Since `shipping_limit_date` should logically fall close to `order_purchase_timestamp` (it's the seller's handling deadline, set at order time), a date this far out is suspicious rather than expected.

In [2]:
shipping_limit_outliers = dfs["order_items"][
    dfs["order_items"]["shipping_limit_date"] > dfs["orders"]["order_purchase_timestamp"].max()
]

print(f"Outlier rows: {len(shipping_limit_outliers)} out of {len(dfs['order_items'])}")
print(f"Unique sellers affected: {shipping_limit_outliers['seller_id'].nunique()}")
print(f"Unique orders affected: {shipping_limit_outliers['order_id'].nunique()}")

display(shipping_limit_outliers[["order_id", "order_item_id", "seller_id", "shipping_limit_date"]])

Outlier rows: 4 out of 112650
Unique sellers affected: 1
Unique orders affected: 3


,order_id,order_item_id,seller_id,shipping_limit_date
8643,13bdf405f961a6deec817d817f5c6624,1,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,1,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22
85729,c2bb89b5c1dd978d507284be78a04cb2,1,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08
85730,c2bb89b5c1dd978d507284be78a04cb2,2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08


In [3]:
outlier_seller_id = shipping_limit_outliers["seller_id"].iloc[0]

seller_all_items = dfs["order_items"][dfs["order_items"]["seller_id"] == outlier_seller_id].copy()

print(f"Total order items from this seller: {len(seller_all_items)}")

seller_all_items["shipping_limit_year"] = seller_all_items["shipping_limit_date"].dt.year

seller_year_counts = (
    seller_all_items
    .groupby("shipping_limit_year")
    .size()
    .reset_index(name="order_item_count")
)

seller_year_counts

Total order items from this seller: 72


,shipping_limit_year,order_item_count
0,2017,31
1,2018,37
2,2020,4


**Data quality finding — `shipping_limit_date` outliers**

4 rows in `order_items` (out of 112,650 total, 0.004%) have a `shipping_limit_date` in 2020 — roughly two years after the last actual order in the dataset (Oct 2018). All 4 outliers belong to a single seller and 3 distinct orders. That seller has 72 order items total in the dataset, and only these 4 fall outside a normal range — confirming this is an isolated data entry error (not a systemic issue with this seller's data or with the column generally).

---
### Investigations Triggered by Numeric Summary

The following investigations were triggered by anomalies observed in the numeric summary table in `03_exploratory_data_analysis.ipynb`.

- Investigating geolocation lat/lng outliers
- Investigating `payment_installments` = 0
- Investigating `product_weight_g` = 0
- Extreme value inspection

#### **Investigating geolocation lat/lng outliers**

The numeric summary shows `geolocation_lat` reaching as high as 45.07 and `geolocation_lng` ranging from -101.47 to 121.11. Brazil's actual geographic bounding box is roughly latitude -34 to +5 and longitude -74 to -34. Coordinates outside this range are geocoding errors, not real locations, and would place points outside Brazil entirely (or even outside South America) on a map. Since this file is intended to power a lat/long heatmap in Tableau, these need to be quantified and understood before deciding how to handle them.

In [4]:
brazil_lat_range = (-34, 6)
brazil_lng_range = (-75, -32)

geo = dfs["geolocation"]

out_of_bounds = geo[
    (geo["geolocation_lat"] < brazil_lat_range[0]) | (geo["geolocation_lat"] > brazil_lat_range[1]) |
    (geo["geolocation_lng"] < brazil_lng_range[0]) | (geo["geolocation_lng"] > brazil_lng_range[1])
]

print(f"Total geolocation rows: {len(geo)}")
print(f"Out-of-bounds rows: {len(out_of_bounds)}")
print(f"Out-of-bounds percentage: {round(len(out_of_bounds) / len(geo) * 100, 4)}%")
print(f"Distinct zip code prefixes affected: {out_of_bounds['geolocation_zip_code_prefix'].nunique()}")

display(out_of_bounds.sort_values("geolocation_lat").head(20))

Total geolocation rows: 1000163
Out-of-bounds rows: 31
Out-of-bounds percentage: 0.0031%
Distinct zip code prefixes affected: 20


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
992584,98780,-36.61,-64.28,santa rosa,RS
993302,98780,-36.60,-64.29,santa rosa,RS
993075,98780,-36.60,-64.29,santa rosa,RS
695375,45936,-34.62,-58.90,itabata,BA
513643,28155,-34.59,-58.73,santa maria,RJ
965687,95130,14.59,121.11,santa lucia do piai,RS
538557,29654,21.66,-101.47,santo antonio do canaa,ES
585242,35179,26.00,-98.08,santana do paraíso,MG
585260,35179,26.00,-98.08,santana do paraiso,MG
387565,18243,28.01,-15.54,bom retiro da esperanca,SP


In [5]:
print(f"Out-of-bounds rows: {len(out_of_bounds)}")
print(f"Out-of-bounds percentage: {round(len(out_of_bounds) / len(geo) * 100, 4)}%")
print(f"Distinct zip code prefixes affected: {out_of_bounds['geolocation_zip_code_prefix'].nunique()}")
print(f"Distinct states affected: {sorted(out_of_bounds['geolocation_state'].unique())}")

Out-of-bounds rows: 31
Out-of-bounds percentage: 0.0031%
Distinct zip code prefixes affected: 20
Distinct states affected: ['AL', 'BA', 'ES', 'MG', 'MT', 'PA', 'PB', 'PR', 'RJ', 'RS', 'SP']


**Finding — geolocation lat/lng outliers (cross-country geocoding errors)**

31 rows (0.0031% of 1,000,163 geolocation rows), spanning 20 distinct zip code prefixes across 11 states, have latitude/longitude coordinates that fall outside Brazil's geographic bounding box (lat -34 to 6, lng -75 to -32). Inspection shows these are not random noise — they consistently land on real-world locations sharing the same city name as the Brazilian town, but in the wrong country.

**Decision:** left as-is in this raw/EDA notebook. Affected rows will be excluded (or have lat/lng nulled, with city/state retained) when building `geolocation_table` in the ETL script, using the same bounding-box check as a validation rule. At 0.003% of rows, this has no material impact on any aggregate geographic analysis, but would visibly misplace a handful of points on a Tableau map if left unfiltered — so it's a cleaning rule worth keeping even though the data volume is trivial.

#### **Investigating payment_installments = 0**

The numeric summary shows `payment_installments` has a minimum of 0. Logically, even a single lump-sum payment should be recorded as 1 installment, so a value of 0 is unexpected. Before deciding how to treat this, we isolate the affected.

In [6]:
zero_installment_payments = dfs["payments"][dfs["payments"]["payment_installments"] == 0]

print(f"Rows with payment_installments = 0: {len(zero_installment_payments)}")
print(f"Out of total payment rows: {len(dfs['payments'])}")

display(zero_installment_payments["payment_type"].value_counts())
display(zero_installment_payments)

Rows with payment_installments = 0: 2
Out of total payment rows: 103886


payment_type
credit_card    2
Name: count, dtype: int64

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [7]:
affected_order_ids = zero_installment_payments["order_id"].tolist()

full_payment_history = dfs["payments"][
    dfs["payments"]["order_id"].isin(affected_order_ids)
].sort_values(["order_id", "payment_sequential"])

display(full_payment_history)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69


**Finding — `payment_installments = 0`**

2 rows in `payments` (out of 103,886) have `payment_installments = 0`, which is logically invalid (even a single payment should register as 1 installment). Both affected rows are `payment_type = credit_card` with `payment_sequential = 2` — but in both cases, no `payment_sequential = 1` row exists for the same order, meaning these are not valid second-leg split payments but appear to be standalone, malformed payment records (a missing first payment, combined with a bad installment value on the surviving row).

**Decision:** left as-is in this notebook. At 2 rows out of 103,886 (0.002%), this has no material impact on aggregate payment analysis. In the ETL script, `payment_installments = 0` will be corrected to 1 (the logical minimum for any recorded payment), and this will be documented as a targeted data-quality fix rather than a row exclusion, since the `payment_value` itself (BRL 58.69 and BRL 129.94) appears valid and shouldn't be dropped from revenue totals.

#### **Investigating product_weight_g = 0**

The numeric summary shows `product_weight_g` has a minimum of 0. A physical product weighing zero grams isn't realistic, so this is likely a data entry gap rather than a genuine value. Before deciding how to handle it, we isolate the affected rows to see how many products are involved and whether their other dimension fields (length, height, width) are also missing or invalid.

In [8]:
zero_weight_products = dfs["products"][dfs["products"]["product_weight_g"] == 0]

print(f"Products with weight = 0: {len(zero_weight_products)}")
print(f"Out of total products: {len(dfs['products'])}")

display(zero_weight_products)

Products with weight = 0: 4
Out of total products: 32951


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.00,529.00,1.00,0.00,30.00,25.00,30.00
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.00,528.00,1.00,0.00,30.00,25.00,30.00
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.00,528.00,1.00,0.00,30.00,25.00,30.00
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.00,528.00,1.00,0.00,30.00,25.00,30.00


**Finding — `product_weight_g = 0`**

4 products (out of 32,951) have `product_weight_g = 0`, which isn't a realistic value for a physical product. All 4 share the same category (bed/bath/table linens) and nearly identical dimensions (~30×25×30 cm), with all other fields (name length, description length, photo count) populated normally. This suggests an isolated data entry gap specific to the weight field for a small batch of similar listings, not a broader data quality issue with the products table.

**Decision:** left as-is in this notebook. At 4 rows out of 32,951 (0.012%), this has negligible impact on aggregate product analysis. In the ETL script, `product_weight_g = 0` will be treated as missing (set to null or imputed using the category median weight) rather than treated as a valid zero, and will be documented. This also matters for freight-cost-related analysis in, since freight_value is influenced by weight — a true zero would understate any weight-to-freight correlation.

#### **Extreme Value Inspection**

The numeric summary flagged large maximum values for `price` (BRL 6,735) and `payment_value` (BRL 13,664). Before trusting these in revenue analysis, we inspect the actual rows behind these extremes to distinguish legitimate high-value transactions from data entry errors.

In [9]:
order_items_with_category = dfs["order_items"].merge(
    dfs["products"][["product_id", "product_category_name"]],
    on="product_id",
    how="left"
).merge(
    dfs["category_translation"],
    on="product_category_name",
    how="left"
)

print("Top 10 items by price:")
display(
    order_items_with_category
    .nlargest(10, "price")[["order_id", "product_category_name_english", "price", "freight_value"]]
)

print("Top 10 payments by payment_value:")
display(
    dfs["payments"]
    .nlargest(10, "payment_value")[["order_id", "payment_type", "payment_installments", "payment_value"]]
)

Top 10 items by price:


,order_id,product_category_name_english,price,freight_value
3556,0812eb902a67711a1cb742b3cdaa65ae,housewares,"6,735.00",194.31
112233,fefacc66af859508bf1a7934eab1e97f,computers,"6,729.00",193.21
107841,f5136e38d1a14a4dbd87dff67da82701,art,"6,499.00",227.66
74336,a96610ab360d42a2e5335a3998b4718a,small_appliances,"4,799.00",151.34
11249,199af31afc78c699f0dbf71fb178d4d4,small_appliances,"4,690.00",74.34
62086,8dbc85d1447242f3b127dda390d56e19,computers,"4,590.00",91.78
29193,426a9742b533fc6fed17d1fd6d143d7e,musical_instruments,"4,399.87",113.45
45843,68101694e5c5dc7330c91e1bbc36214f,consoles_games,"4,099.99",75.27
78310,b239ca7cd485940b31882363b52e6674,sports_leisure,"4,059.00",104.51
59137,86c4eab1571921a6a6e248ed312f5a5a,watches_gifts,"3,999.90",17.01


Top 10 payments by payment_value:


,order_id,payment_type,payment_installments,payment_value
52107,03caa2c082116e1d31e67e9ae3700499,credit_card,1,"13,664.08"
34370,736e1922ae60d0d6a89247b851902527,boleto,1,"7,274.88"
41419,0812eb902a67711a1cb742b3cdaa65ae,credit_card,8,"6,929.31"
49581,fefacc66af859508bf1a7934eab1e97f,boleto,1,"6,922.21"
85539,f5136e38d1a14a4dbd87dff67da82701,boleto,1,"6,726.66"
62409,2cc9089445046817a7539d90805e6e5a,boleto,1,"6,081.54"
43232,a96610ab360d42a2e5335a3998b4718a,credit_card,10,"4,950.34"
70320,b4c4b76c642808cbe472a32b86cddc95,credit_card,5,"4,809.44"
6440,199af31afc78c699f0dbf71fb178d4d4,credit_card,8,"4,764.34"
67546,8dbc85d1447242f3b127dda390d56e19,credit_card,8,"4,681.78"


In [10]:
top_payment_order = dfs["payments"].nlargest(1, "payment_value")["order_id"].iloc[0]

display(
    order_items_with_category[order_items_with_category["order_id"] == top_payment_order][
        ["order_id", "product_category_name_english", "price", "freight_value"]
    ]
)

,order_id,product_category_name_english,price,freight_value
1647,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1648,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1649,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1650,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1651,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1652,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1653,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01
1654,03caa2c082116e1d31e67e9ae3700499,fixed_telephony,"1,680.00",28.01


**Finding — extreme value inspection (`price` and `payment_value`)**

Top 10 items by price range from BRL 3,999 to BRL 6,735 across categories including `computers`, `small_appliances`, `musical_instruments`, `consoles_games`, and `watches_gifts` — all plausible given Brazil's high import duties on electronics and luxury goods.

The highest payment value (BRL 13,664, single lump-sum credit card) resolves to a bulk purchase of 8 identical `fixed_telephony` items at BRL 1,680 each — a legitimate B2B-style transaction, not a data entry error. All other top payment values correspond directly to high-priced items already identified in the items table.

**Decision:** no cleaning applied. All extreme values inspected are explainable and consistent with real transaction patterns. These rows will be retained as-is in the warehouse and included in all revenue metrics.

---
### Investigations Triggered by Categorical Summary

The following investigations were triggered by anomalies observed in the categorical summary table in `03_exploratory_data_analysis.ipynb`.

- Investigating product category translation mismatch

#### **Investigating product category translation mismatch**

`products.product_category_name` has 73 unique values, but `category_translation.product_category_name` has only 71. This means at least one (likely two) category names used in real product records have no matching English translation — which would produce nulls in after the joins in ETL. Before deciding how to handle it, we identify exactly which category names are missing.

In [11]:
product_categories = set(dfs["products"]["product_category_name"].dropna())
translated_categories = set(dfs["category_translation"]["product_category_name"])

missing_translations = product_categories - translated_categories

print(f"Categories in products with no translation: {len(missing_translations)}")
print(missing_translations)

affected_product_count = dfs["products"]["product_category_name"].isin(missing_translations).sum()
print(f"Products affected: {affected_product_count}")

Categories in products with no translation: 2
{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}
Products affected: 13


**Finding — 2 product categories missing English translation**

Two categories — `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` — have no English translation, affecting 13 products total. Both are legitimate category names (gaming PCs; portable kitchen/food prep appliances), this is simply a gap in the translation lookup file, not a data entry error in `products`.

**Decision:** left as-is in this notebook. In the ETL script, the left join from `products` to `category_translation` null for these 13 products unless a manual translation mapping is added for these two categories. This will be documented and the manual mapping (if added) will be called out explicitly as a judgment call rather than sourced from Olist's original data.

---
### Systematic Quality Checks

The following checks were performed proactively as part of standard data validation practice, independent of any specific anomaly flagged in another notebook.

- Identifier consistency
- Investigating `review_id` duplication
- Investigating missing payment record
- Referential integrity
- Temporal logic consistency
- Investigating delivered orders missing delivery timestamp
- Orders without items
- Negative value check

#### **Identifier Consistency**

Olist's tables are connected through several shared `_id` columns (`order_id`, `customer_id`, `product_id`, `seller_id`, `review_id`). We first confirm each id column's basic shape: is it unique within its own table, and how many distinct tables reference it.

In [12]:
id_rows = []

for name, df in dfs.items():
    id_columns = [col for col in df.columns if col.lower().endswith("_id") or col.lower() == "id"]

    for col in id_columns:
        id_rows.append({
            "dataset": name,
            "id_column": col,
            "rows": len(df),
            "non_null_count": df[col].notna().sum(),
            "unique_values": df[col].nunique(dropna=True),
            "duplicate_values": df[col].duplicated().sum(),
            "appears_unique_in_table": df[col].notna().sum() == df[col].nunique(dropna=True)
        })

id_profile = pd.DataFrame(id_rows)
id_profile

,dataset,id_column,rows,non_null_count,unique_values,duplicate_values,appears_unique_in_table
0,orders,order_id,99441,99441,99441,0,True
1,orders,customer_id,99441,99441,99441,0,True
2,customers,customer_id,99441,99441,99441,0,True
3,customers,customer_unique_id,99441,99441,96096,3345,False
4,order_items,order_id,112650,112650,98666,13984,False
5,order_items,order_item_id,112650,112650,21,112629,False
6,order_items,product_id,112650,112650,32951,79699,False
7,order_items,seller_id,112650,112650,3095,109555,False
8,payments,order_id,103886,103886,99440,4446,False
9,reviews,review_id,99224,99224,98410,814,False


#### **Investigating review_id duplication**

The identifier consistency check showed `review_id` has 814 duplicate values in the `reviews` table, even though the earlier duplicate-row check found zero fully duplicated rows. That combination means the same `review_id` is attached to rows that differ in at least one other column — which would violate `review_id` as a primary key. Before deciding how we should handle this, we isolate the affected `review_id`s to see what's actually different between the duplicate occurrences.

In [13]:
duplicated_review_ids = dfs["reviews"][
    dfs["reviews"]["review_id"].duplicated(keep=False)
].sort_values("review_id")

print(f"Rows involved: {len(duplicated_review_ids)}")
print(f"Distinct review_id values affected: {duplicated_review_ids['review_id'].nunique()}")

display(duplicated_review_ids.head(20))

Rows involved: 1603
Distinct review_id values affected: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


**Finding — `review_id` duplication explained**

814 `review_id` values (1,603 rows total) appear attached to more than one `order_id`. In every sampled case, the review content (score, comment, timestamps) is identical across the duplicates — only `order_id` differs. This matches a known Olist behavior: when a customer's cart spans multiple sellers, Olist splits the checkout into multiple `order_id`s, but only one review is collected for the whole session and gets linked to every resulting order.

**Decision:** keep these rows as-is in `reviews_table` (grain = one row per `order_id`, not strictly one row per unique review), since the table needs to join cleanly to `order_items` via `order_id`. Any metric counting "number of reviews received" must use `COUNT(DISTINCT review_id)`, not row count, to avoid overcounting by ~1,600.

#### **Investigating missing payment record**

The identifier consistency check showed `payments.order_id` has 99,440 unique values against 99,441 total orders — implying exactly one order has no payment record at all. Before deciding whether this matters, we check which order it is and what its status is.

In [14]:
orders_without_payment = set(dfs["orders"]["order_id"]) - set(dfs["payments"]["order_id"])

print(f"Orders without a payment record: {len(orders_without_payment)}")

missing_payment_order = dfs["orders"][dfs["orders"]["order_id"].isin(orders_without_payment)]

display(missing_payment_order)

Orders without a payment record: 1


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [15]:
missing_payment_items = dfs["order_items"][
    dfs["order_items"]["order_id"] == "bfbd0f9bdef84302105ad712db648a6c"
]

display(missing_payment_items)

print(f"Order value (price + freight): {(missing_payment_items['price'] + missing_payment_items['freight_value']).sum():.2f}")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
84389,bfbd0f9bdef84302105ad712db648a6c,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84390,bfbd0f9bdef84302105ad712db648a6c,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83
84391,bfbd0f9bdef84302105ad712db648a6c,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83


Order value (price + freight): 143.46


**Finding — order with no payment record**

Order `bfbd0f9bdef84302105ad712db648a6c` (status: `delivered`, purchased 2016-09-15) has 3 order items totaling **BRL 143.46** (price + freight) but zero matching rows in `payments`. This is one of the earliest orders in the dataset (purchase date is near the dataset's overall minimum of 2016-09-04), suggesting a possible early-platform logging gap rather than a systemic issue — `order_approved_at` is also identical to `order_purchase_timestamp`, which is atypical and supports this being an early/edge-case record.

**Decision:** left as-is in this notebook. Any SQL view that calculates total revenue via `payments.payment_value` will silently exclude this order's BRL 143.46. If total revenue is reported from `order_items.price` instead, this order is naturally included. This discrepancy (revenue-by-payments vs revenue-by-order-items) will be documented, so the two metrics aren't expected to match exactly, and the BRL 143.46 gap is explained rather than mistaken for a bug.

#### **Referential Integrity**

Identifier consistency above confirms each key's shape within its own table, but not whether foreign keys actually resolve to real records in the parent table. Here we check the relationships our star schema design depends on — every foreign key value in a child table should exist in its corresponding parent table's primary key. Any orphaned values found here would mean rows that can't be cleanly joined once loaded into the warehouse, and would need a decision (drop, null out, or investigate further).

In [16]:
referential_checks = [
    {"child": "order_items", "child_key": "order_id", "parent": "orders", "parent_key": "order_id"},
    {"child": "order_items", "child_key": "product_id", "parent": "products", "parent_key": "product_id"},
    {"child": "order_items", "child_key": "seller_id", "parent": "sellers", "parent_key": "seller_id"},
    {"child": "orders", "child_key": "customer_id", "parent": "customers", "parent_key": "customer_id"},
    {"child": "payments", "child_key": "order_id", "parent": "orders", "parent_key": "order_id"},
    {"child": "reviews", "child_key": "order_id", "parent": "orders", "parent_key": "order_id"}
]

referential_summary = []

for check in referential_checks:
    child_values = set(dfs[check["child"]][check["child_key"]].dropna())
    parent_values = set(dfs[check["parent"]][check["parent_key"]].dropna())

    orphaned = child_values - parent_values

    referential_summary.append({
        "child_table": check["child"],
        "child_key": check["child_key"],
        "parent_table": check["parent"],
        "parent_key": check["parent_key"],
        "child_distinct_values": len(child_values),
        "orphaned_values": len(orphaned),
        "orphaned_percentage": round(len(orphaned) / len(child_values) * 100, 4) if child_values else 0
    })

referential_summary_df = pd.DataFrame(referential_summary)

referential_summary_df

,child_table,child_key,parent_table,parent_key,child_distinct_values,orphaned_values,orphaned_percentage
0,order_items,order_id,orders,order_id,98666,0,0.00
1,order_items,product_id,products,product_id,32951,0,0.00
2,order_items,seller_id,sellers,seller_id,3095,0,0.00
3,orders,customer_id,customers,customer_id,99441,0,0.00
4,payments,order_id,orders,order_id,99440,0,0.00
5,reviews,order_id,orders,order_id,98673,0,0.00


#### **Temporal Logic Consistency**

Order timestamps should follow a logical sequence: `order_purchase_timestamp` → `order_approved_at` → `order_delivered_carrier_date` → `order_delivered_customer_date`. Rows where a later event is recorded before an earlier one are timestamp sequencing errors, not just nulls, and would produce negative or incorrect delivery durations in time-based analysis.

In [17]:
orders = dfs["orders"].copy()

temporal_checks = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date")
]

temporal_summary = []

for earlier, later in temporal_checks:
    both_present = orders[[earlier, later]].dropna()
    violations = both_present[both_present[earlier] > both_present[later]]

    temporal_summary.append({
        "earlier_event": earlier,
        "later_event": later,
        "rows_with_both_timestamps": len(both_present),
        "sequence_violations": len(violations),
        "violation_percentage": round(len(violations) / len(both_present) * 100, 4) if len(both_present) > 0 else 0
    })

pd.DataFrame(temporal_summary)

,earlier_event,later_event,rows_with_both_timestamps,sequence_violations,violation_percentage
0,order_purchase_timestamp,order_approved_at,99281,0,0.00
1,order_approved_at,order_delivered_carrier_date,97644,1359,1.39
2,order_delivered_carrier_date,order_delivered_customer_date,96475,23,0.02
3,order_purchase_timestamp,order_delivered_customer_date,96476,0,0.00


**Finding — timestamp sequence violations**

Two pairs in the order lifecycle show sequencing inconsistencies:

- `order_approved_at` → `order_delivered_carrier_date`: 1,359 violations (1.39%) where the carrier pickup was recorded before payment approval. Likely a real business process pattern (pre-shipment by marketplace sellers) rather than data corruption, but the sequence cannot be guaranteed.
- `order_delivered_carrier_date` → `order_delivered_customer_date`: 23 violations (0.02%), likely genuine logging errors.

The end-to-end check (`order_purchase_timestamp` → `order_delivered_customer_date`) shows zero violations, confirming the data is reliable at the order level.

**Decision:** no cleaning applied. Any metric involving delivery duration will use `order_delivered_customer_date − order_purchase_timestamp` (end-to-end) rather than relying on intermediate timestamps. This will be noted, so downstream analysts don't build metrics on `order_delivered_carrier_date` without understanding the 1.39% sequencing caveat.

#### **Delivered Orders Missing a Delivery Timestamp**

`order_status = delivered` implies the customer physically received the order, so `order_delivered_customer_date` should always be populated for these rows. A delivered order with a null delivery timestamp is a logical inconsistency — it would silently drop from any delivery-time calculation in Analysis Phase without explanation.

In [18]:
delivered_orders = dfs["orders"][dfs["orders"]["order_status"] == "delivered"]

missing_delivery_date = delivered_orders[delivered_orders["order_delivered_customer_date"].isna()]

print(f"Total delivered orders: {len(delivered_orders)}")
print(f"Delivered orders missing customer delivery date: {len(missing_delivery_date)}")
print(f"Percentage: {round(len(missing_delivery_date) / len(delivered_orders) * 100, 4)}%")

display(missing_delivery_date[["order_id", "order_status", "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]].head(20))

Total delivered orders: 96478
Delivered orders missing customer delivery date: 8
Percentage: 0.0083%


,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT


**Finding — delivered orders missing `order_delivered_customer_date`**

8 orders (out of 96,478 delivered, 0.008%) have `order_status = delivered` but a null `order_delivered_customer_date`. Unlike the missing payment record (which dated to the platform's earliest days), these span 2017–2018, suggesting isolated carrier scan failures rather than a systemic early-platform logging issue. One order (purchased 2017-05-25) is also missing `order_delivered_carrier_date`, making it the most logically inconsistent record — marked delivered with no delivery evidence at all.

**Decision:** left as-is. At 8 rows out of 96,478 (0.008%), these will silently drop from any delivery-time calculation that requires a non-null `order_delivered_customer_date` — this is acceptable given the volume. The missing carrier+customer date order will be noted as an unresolvable record. No imputation or status correction will be applied.

#### **Orders Without Items**

Every order should have at least one corresponding row in `order_items`.

In [19]:
orders_with_items = set(dfs["order_items"]["order_id"])
all_orders = set(dfs["orders"]["order_id"])

orders_without_items = all_orders - orders_with_items

print(f"Total orders: {len(all_orders)}")
print(f"Orders with at least one item: {len(orders_with_items)}")
print(f"Orders with no items: {len(orders_without_items)}")

missing_items_orders = dfs["orders"][dfs["orders"]["order_id"].isin(orders_without_items)]

display(missing_items_orders["order_status"].value_counts())
display(missing_items_orders)

Total orders: 99441
Orders with at least one item: 98666
Orders with no items: 775


order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
266,8e24261a7e58791d10cb1bf9da94df5c,64a254d30eed42cd0e6c36dddb88adf0,unavailable,2017-11-16 15:09:28,2017-11-16 15:26:57,NaT,NaT,2017-12-05
586,c272bcd21c287498b4883c7512019702,9582c5bbecc65eb568e2c1d839b5cba1,unavailable,2018-01-31 11:31:37,2018-01-31 14:23:50,NaT,NaT,2018-02-16
687,37553832a3a89c9b2db59701c357ca67,7607cd563696c27ede287e515812d528,unavailable,2017-08-14 17:38:02,2017-08-17 00:15:18,NaT,NaT,2017-09-05
737,d57e15fb07fd180f06ab3926b39edcd2,470b93b3f1cde85550fc74cd3a476c78,unavailable,2018-01-08 19:39:03,2018-01-09 07:26:08,NaT,NaT,2018-02-06
1130,00b1cb0320190ca0daa2c88b35206009,3532ba38a3fd242259a514ac2b6ae6b6,canceled,2018-08-28 15:26:39,NaT,NaT,NaT,2018-09-12
...,...,...,...,...,...,...,...,...
99252,aaab15da689073f8f9aa978a390a69d1,df20748206e4b865b2f14a5eabbfcf34,unavailable,2018-01-16 14:27:59,2018-01-17 03:37:34,NaT,NaT,2018-02-06
99283,3a3cddda5a7c27851bd96c3313412840,0b0d6095c5555fe083844281f6b093bb,canceled,2018-08-31 16:13:44,NaT,NaT,NaT,2018-10-01
99347,a89abace0dcc01eeb267a9660b5ac126,2f0524a7b1b3845a1a57fcf3910c4333,canceled,2018-09-06 18:45:47,NaT,NaT,NaT,2018-09-27
99348,a69ba794cc7deb415c3e15a0a3877e69,726f0894b5becdf952ea537d5266e543,unavailable,2017-08-23 16:28:04,2017-08-28 15:44:47,NaT,NaT,2017-09-15


**Finding — orders without items**

775 orders (out of 99,441, 0.78%) have no corresponding rows in `order_items`. All are non-fulfilled statuses: `unavailable` (603), `canceled` (164), `created` (5), `invoiced` (2), `shipped` (1). No `delivered` orders are missing items, confirming this is expected behavior — orders that never reached fulfillment simply never had items attached, rather than a data loss issue.

**Decision:** left as-is. These 775 orders will naturally be excluded from any `order_items`-based metric (revenue, product analysis, seller analysis) since they have no item rows to join to. They will remain in `order_items`'s source data and be visible in order-status distribution counts, which is correct — they are real orders, just unfulfilled ones. No special handling needed in ETL Phase beyond documenting that order-level counts (from `orders`) and item-level counts (from `order_items`) will not match by design.

#### **Negative Value Check**

The numeric summary showed minimums at or above zero for price, freight, payment, and product dimension columns, but this was incidental — the summary table was never designed as a negative value check. Here we make it explicit and deliberate, so it's documented as a confirmed validation rather than an assumed one.

In [20]:
negative_checks = [
    {"dataset": "order_items", "column": "price"},
    {"dataset": "order_items", "column": "freight_value"},
    {"dataset": "payments", "column": "payment_value"},
    {"dataset": "payments", "column": "payment_installments"},
    {"dataset": "products", "column": "product_weight_g"},
    {"dataset": "products", "column": "product_length_cm"},
    {"dataset": "products", "column": "product_height_cm"},
    {"dataset": "products", "column": "product_width_cm"},
]

negative_summary = []

for check in negative_checks:
    col_data = dfs[check["dataset"]][check["column"]]
    negative_count = (col_data < 0).sum()

    negative_summary.append({
        "dataset": check["dataset"],
        "column": check["column"],
        "negative_count": negative_count,
        "negative_percentage": round(negative_count / col_data.notna().sum() * 100, 4)
    })

pd.DataFrame(negative_summary)

,dataset,column,negative_count,negative_percentage
0,order_items,price,0,0.00
1,order_items,freight_value,0,0.00
2,payments,payment_value,0,0.00
3,payments,payment_installments,0,0.00
4,products,product_weight_g,0,0.00
5,products,product_length_cm,0,0.00
6,products,product_height_cm,0,0.00
7,products,product_width_cm,0,0.00


**Finding — no negative values confirmed**

All numeric columns representing physical quantities (price, freight_value, payment_value, payment_installments, product_weight_g, product_length_cm, product_height_cm, product_width_cm) were explicitly checked for negative values. Zero negative values found across all columns. This confirms the numeric floor constraints in the warehouse DDL (`NOT NULL` and positive-value assumptions) are safe to apply without a defensive cleaning step in the ETL pipeline.

## Summary

This notebook systematically validated the trustworthiness of the Olist dataset through 12 investigations across four categories: anomalies surfaced during data assessment, anomalies surfaced during exploratory analysis, and proactive quality checks covering identifiers, referential integrity, temporal logic, and completeness.

| # | Finding | Severity | Decision |
|---|---|---|---|
| 1 | `shipping_limit_date` — 4 rows from 1 seller dated 2020 | Low | Cap or null out in ETL Phase |
| 2 | Geolocation lat/lng — 31 rows outside Brazil's bounding box | Low | Exclude using bounding box filter in ETL Phase |
| 3 | `payment_installments` = 0 — 2 rows, no sequential = 1 | Low | Set to 1 in ETL Phase |
| 4 | `product_weight_g` = 0 — 4 products, same category | Low | Null out or impute using category median in ETL Phase |
| 5 | Extreme values in `price` and `payment_value` — confirmed legitimate | None | No action needed |
| 6 | 2 product categories missing English translation — 13 products affected | Low | Add manual mapping in ETL Phase |
| 7 | `review_id` fan-out — 814 IDs attached to multiple orders | None | Use `COUNT(DISTINCT review_id)` in Analysis Phase |
| 8 | 1 delivered order missing payment record — BRL 143.46 | Low | Document revenue discrepancy in `docs/05_etl_methodology.md` |
| 9 | 775 orders with no items — all non-fulfilled statuses | None | Expected behavior, document in `docs/05_etl_methodology.md` |
| 10 | Timestamp sequence violations — 1,359 carrier before approval, 23 carrier after delivery | Low | Use end-to-end delivery time only in Analysis Phase |
| 11 | 8 delivered orders missing `order_delivered_customer_date` | Low | Accept silent exclusion from delivery-time metrics |
| 12 | No negative values in any numeric quantity column | None | Confirmed, no action needed |

**Overall assessment:** the Olist dataset is of high quality for a real-world e-commerce dataset. No finding is severe enough to materially distort aggregate analysis. All cleaning rules will be implemented in the ETL pipeline and will be documented.

<hr style="height:2px; background-color:black; border:none;">